# Course 8: Research Engineering Interviews
## Session 8: Deep learning and LLM fundamentals

**Goal:** make quick estimates of parameters, memory and compute, and check them in code.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** the tests and reference solutions here were run once on a CPU (November 2026). Cells marked TODO are stubs for you; try them under a timer before opening the reference solutions at the bottom.

### Do each on paper first, then run the cell

In [ ]:
GB = 1e9
def weights_gb(params, bits=16): return params * bits / 8 / GB
def adam_training_gb(params, bits=32): return 4 * params * bits / 8 / GB      # weights, grads, two moments
def lora_params(d, k, r): return r * (d + k)
def attention_scores_gb(batch, heads, T, bits=16): return batch * heads * T * T * bits / 8 / GB
def gpt_block_params(layers, d): return layers * 12 * d * d

print("1. 3B weights at 16-bit:", weights_gb(3e9), "GB")
print("2. Adam state for 1B at 32-bit:", adam_training_gb(1e9), "GB")
print("3. LoRA rank 8 on 4096 x 4096:", lora_params(4096, 4096, 8))
print("4. Attention scores, batch 8, 12 heads, 2048 tokens:", round(attention_scores_gb(8, 12, 2048), 2), "GB")
print("5. Embedding 50,000 x 768:", 50000 * 768 / 1e6, "M params")
print("6. Blocks, 12 layers at width 768:", round(gpt_block_params(12, 768) / 1e6, 1), "M params")

### The 24 GB question

In [ ]:
print("7B full fine-tune with Adam, 32-bit:", adam_training_gb(7e9), "GB  -> does not fit in 24 GB")
print("7B frozen base at 4-bit:", weights_gb(7e9, 4), "GB  -> fits, with LoRA adapters on top")

### Compute rule of thumb
About 6 floating-point operations per parameter per token for one forward and backward pass.

In [ ]:
def train_flops(params, tokens): return 6 * params * tokens
flops = train_flops(100e6, 1e9)
print(f"100M model on 1B tokens: {flops:.1e} FLOPs")
gpu_flops_per_s = 1e13          # an assumption: state yours aloud in an interview
print("at 1e13 useful FLOP/s:", round(flops / gpu_flops_per_s / 3600, 1), "GPU-hours")

### TODO
- Write five new estimate questions with worked answers and swap with a partner.
- Give a one-sentence reason for each: residuals, layer norm, warm-up, weight decay, dropout, positional information, square-root scaling, weight tying, gradient clipping, mixed precision.